In [ ]:
!mkdir data
!mkdir models
!mkdir models/opus_mt_en_es_trained
!mkdir models/t5_small_trained

: 

In [4]:
!pip install evaluate
!pip install bert_score

In [5]:
import pandas as pd
from evaluate import load
from datasets import Dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

In [6]:
def train(model, train_loader, optimizer, epochs=5, device='cuda'):
    model = model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

In [7]:
data = pd.read_csv('data/en_es_corpus.txt', sep='\t', header=None)
data = data[[0, 1]]
data.columns = ['EN', 'ES']
dataset = data[-1000:]
sentences_en = dataset['EN'].values.tolist()
sentences_es = dataset['ES'].values.tolist()

In [8]:
data

,EN,ES
0,Go.,Ve.
1,Go.,Vete.
2,Go.,Vaya.
3,Go.,Váyase.
4,Hi.,Hola.
...,...,...
139008,A carbon footprint is the amount of carbon dio...,Una huella de carbono es la cantidad de contam...
139009,Since there are usually multiple websites on a...,Como suele haber varias páginas web sobre cual...
139010,"If you want to sound like a native speaker, yo...","Si quieres sonar como un hablante nativo, debe..."
139011,It may be impossible to get a completely error...,Puede que sea imposible obtener un corpus comp...


In [29]:
model_name = 'Helsinki-NLP/opus-mt-en-es'
# model_name = 'T5-small'

In [30]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (2 outputs left out)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [31]:
inputs = tokenizer(sentences_en, max_length=25, truncation=True)
outputs = tokenizer(text_target=sentences_es, max_length=25, truncation=True)

train_set = Dataset.from_dict({'input_ids': inputs['input_ids'],
                              'attention_mask': inputs['attention_mask'],
                              'labels': outputs['input_ids']})

In [32]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, collate_fn=data_collator)
optimizer = AdamW(model.parameters(), lr=0.000001)

In [87]:
train(model, train_loader, optimizer, 500, device='cuda')

Epoch 1/500, Loss: 4.9487
Epoch 2/500, Loss: 4.8989
Epoch 3/500, Loss: 4.8614
Epoch 4/500, Loss: 4.8535
Epoch 5/500, Loss: 4.8336
Epoch 6/500, Loss: 4.7854
Epoch 7/500, Loss: 4.7730
Epoch 8/500, Loss: 4.7459
Epoch 9/500, Loss: 4.7124
Epoch 10/500, Loss: 4.7284
Epoch 11/500, Loss: 4.6830
Epoch 12/500, Loss: 4.6646
Epoch 13/500, Loss: 4.6567
Epoch 14/500, Loss: 4.6451
Epoch 15/500, Loss: 4.6110
Epoch 16/500, Loss: 4.6047
Epoch 17/500, Loss: 4.5802
Epoch 18/500, Loss: 4.5879
Epoch 19/500, Loss: 4.5454
Epoch 20/500, Loss: 4.5360
Epoch 21/500, Loss: 4.5021
Epoch 22/500, Loss: 4.4964
Epoch 23/500, Loss: 4.4828
Epoch 24/500, Loss: 4.4578
Epoch 25/500, Loss: 4.4623
Epoch 26/500, Loss: 4.4576
Epoch 27/500, Loss: 4.4217
Epoch 28/500, Loss: 4.4238
Epoch 29/500, Loss: 4.3990
Epoch 30/500, Loss: 4.3934
Epoch 31/500, Loss: 4.4032
Epoch 32/500, Loss: 4.3864
Epoch 33/500, Loss: 4.3712
Epoch 34/500, Loss: 4.3602
Epoch 35/500, Loss: 4.3332
Epoch 36/500, Loss: 4.3536
Epoch 37/500, Loss: 4.3339
Epoch 38/5

In [16]:
# model.save_pretrained(f'models/opus_mt_en_es_trained')
model.save_pretrained(f'models/t5_small_trained')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [33]:
model2 = AutoModelForSeq2SeqLM.from_pretrained(f'models/opus_mt_en_es_trained')
# model2 = AutoModelForSeq2SeqLM.from_pretrained(f'models/t5_small_trained')

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [34]:
input_sentence = data['EN'][30000]
input_sentence

"It's hard to justify."

In [35]:
tokens = tokenizer([input_sentence], return_tensors='pt').to('cuda')
tokens

{'input_ids': tensor([[   94,    20,     9,  2170,    13, 14990,     3,     0]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [36]:
out = model2.to('cuda').generate(**tokens, max_length=25)
out

tensor([[65000,   177,  2149,     4, 12449,     3,     0]], device='cuda:0')

In [37]:
predicted_sentence = tokenizer.decode(out[0], skip_special_tokens=True)
predicted_sentence

'Es difícil de justificar.'

In [91]:
tokens = tokenizer([input_sentence], return_tensors='pt').to('cuda')
out = model2.to('cuda').generate(**tokens, max_length=25)
predicted_sentence = tokenizer.decode(out[0], skip_special_tokens=True)

In [38]:
predicted_sentence

'Es difícil de justificar.'

In [39]:
reference_sentence = data['ES'][30000]
reference_sentence

'Es difícil de justificar.'

In [24]:
bleu = load('bleu')

In [40]:
bleu.compute(predictions=[predicted_sentence], references=[reference_sentence])

{'bleu': 1.0,
 'precisions': [1.0, 1.0, 1.0, 1.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0,
 'translation_length': 5,
 'reference_length': 5}

In [26]:
bertscore = load('bertscore')

In [41]:
bertscore.compute(predictions=[predicted_sentence], references=[reference_sentence], model_type='distilroberta-base')

{'precision': [1.0],
 'recall': [1.0],
 'f1': [1.0],
 'hashcode': 'distilroberta-base_L5_no-idf_version=0.3.12(hug_trans=5.20.2)'}